# Coordinate Generation Analysis
Analyze model confidence and pixel attribution during bbox coordinate generation.

1. **Logit Confidence** - How confident is the model when selecting each coordinate?
2. **Pixel Saliency** - Which pixels drive each coordinate prediction?

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import json, re
from pathlib import Path
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('/efs/user_folders/dnshalam/datasets/lvis')
model_name = 'Qwen/Qwen3-VL-8B-Instruct'
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map='auto',
)
model.eval()

with open(DATA_DIR / 'lvis_validation.json') as f:
    data = json.load(f)
print(f'Loaded model and {len(data)} samples')

In [ ]:
# ── Generate with logit recording ──

def generate_with_logits(model, processor, image, prompt_text, max_new_tokens=128):
    """Generate tokens, recording full logit distributions at each step."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt_text},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    
    input_ids = inputs['input_ids'].clone()
    static_inputs = {k: v for k, v in inputs.items() if k not in ('input_ids', 'attention_mask')}
    n_input = input_ids.shape[1]
    
    records = []
    for step in range(max_new_tokens):
        attn_mask = torch.ones_like(input_ids)
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attn_mask, **static_inputs)
        
        logits = out.logits[0, -1, :]  # (vocab_size,)
        probs = F.softmax(logits.float(), dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
        
        next_tok = logits.argmax(dim=-1)
        top_k_vals, top_k_ids = torch.topk(probs, k=10)
        
        records.append({
            'token_id': next_tok.item(),
            'token_str': processor.tokenizer.decode(next_tok.item()),
            'prob': probs[next_tok].item(),
            'entropy': entropy,
            'top_k_tokens': [processor.tokenizer.decode(t.item()) for t in top_k_ids],
            'top_k_probs': top_k_vals.cpu().numpy(),
        })
        
        input_ids = torch.cat([input_ids, next_tok.unsqueeze(0).unsqueeze(0)], dim=-1)
        if step == 0:
            static_inputs = {k: v for k, v in static_inputs.items()
                           if k not in ('pixel_values', 'pixel_values_videos', 'image_grid_thw', 'video_grid_thw')}
        if next_tok.item() == model.config.eos_token_id:
            break
    
    return records, n_input, inputs

In [ ]:
def find_bbox_token_ranges(records):
    """Find token indices for each coordinate in bbox_2d output."""
    tokens = [r['token_str'] for r in records]
    full_text = ''.join(tokens)
    
    m = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]', full_text)
    if not m:
        print(f'No bbox found in: {full_text[:200]}')
        return None
    
    coord_names = ['x1', 'y1', 'x2', 'y2']
    coord_ranges = {}  # name -> list of step indices
    
    char_pos = 0
    for step_idx, tok_str in enumerate(tokens):
        tok_start = char_pos
        tok_end = char_pos + len(tok_str)
        for i, name in enumerate(coord_names):
            coord_start = m.start(i + 1)
            coord_end = m.end(i + 1)
            if tok_start < coord_end and tok_end > coord_start:
                if name not in coord_ranges:
                    coord_ranges[name] = []
                coord_ranges[name].append(step_idx)
        char_pos = tok_end
    
    return coord_ranges

In [ ]:
# ── Run on a sample ──
SAMPLE_IDX = 0  # <-- change to explore

item = data[SAMPLE_IDX]
prompt_text = item['conversations'][0]['value'].replace('<image>\n', '')
image = Image.open(item['image']).convert('RGB')

print(f'Prompt: {prompt_text}')
records, n_input, inputs = generate_with_logits(model, processor, image, prompt_text)
gen_text = ''.join(r['token_str'] for r in records)
print(f'Generated: {gen_text[:200]}')

coord_ranges = find_bbox_token_ranges(records)
print(f'Coordinate token ranges: {coord_ranges}')

## 1. Logit Confidence per Coordinate

In [ ]:
if coord_ranges:
    coord_names = ['x1', 'y1', 'x2', 'y2']
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    
    for col, name in enumerate(coord_names):
        if name not in coord_ranges:
            continue
        steps = coord_ranges[name]
        
        # Top row: probability of chosen token at each digit
        ax = axes[0, col]
        probs = [records[s]['prob'] for s in steps]
        digit_labels = [records[s]['token_str'] for s in steps]
        bars = ax.bar(range(len(steps)), probs, color='#2196F3')
        ax.set_xticks(range(len(steps)))
        ax.set_xticklabels(digit_labels, fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1)
        ax.set_title(f'{name} - Token Probability', fontsize=12)
        ax.set_ylabel('P(chosen token)')
        for bar, p in zip(bars, probs):
            ax.text(bar.get_x() + bar.get_width()/2, p + 0.02, f'{p:.2f}', ha='center', fontsize=9)
        
        # Bottom row: top-5 alternatives at first digit
        ax = axes[1, col]
        s = steps[0]  # first digit is most interesting
        top_tokens = records[s]['top_k_tokens'][:5]
        top_probs = records[s]['top_k_probs'][:5]
        colors = ['#4CAF50'] + ['#FF9800'] * 4
        ax.barh(range(len(top_tokens)-1, -1, -1), top_probs, color=colors)
        ax.set_yticks(range(len(top_tokens)-1, -1, -1))
        ax.set_yticklabels([repr(t) for t in top_tokens], fontsize=9)
        ax.set_title(f'{name} - Top-5 Alternatives (1st digit)', fontsize=11)
        ax.set_xlabel('Probability')
    
    fig.suptitle(f'Coordinate Confidence: {prompt_text}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Entropy across all generated tokens (lower = more confident)
if coord_ranges:
    fig, ax = plt.subplots(1, 1, figsize=(16, 4))
    entropies = [r['entropy'] for r in records]
    ax.plot(entropies, color='#607D8B', alpha=0.7)
    
    # Highlight coordinate tokens
    colors = {'x1': '#F44336', 'y1': '#2196F3', 'x2': '#4CAF50', 'y2': '#FF9800'}
    for name, steps in coord_ranges.items():
        for s in steps:
            ax.axvline(s, color=colors[name], alpha=0.5, lw=2)
        ax.scatter(steps, [entropies[s] for s in steps], color=colors[name], 
                   s=80, zorder=5, label=name)
    
    ax.set_xlabel('Generation Step')
    ax.set_ylabel('Entropy (lower = more confident)')
    ax.set_title('Token Entropy During Generation')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Pixel Saliency (Gradient × Input)
Which pixels drive each coordinate prediction?

In [ ]:
def compute_pixel_saliency(model, processor, image, prompt_text, target_step, records):
    """Compute gradient-based pixel saliency for a specific generation step."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt_text},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    
    generated_ids = torch.tensor([[r['token_id'] for r in records[:target_step + 1]]], device=model.device)
    input_ids = torch.cat([inputs['input_ids'], generated_ids], dim=-1)
    attn_mask = torch.ones_like(input_ids)
    pixel_values = inputs['pixel_values'].clone().detach().requires_grad_(True)
    static_inputs = {k: v for k, v in inputs.items()
                     if k not in ('input_ids', 'attention_mask', 'pixel_values')}
    
    out = model(input_ids=input_ids, attention_mask=attn_mask,
                pixel_values=pixel_values, **static_inputs)
    
    logits = out.logits[0, -1, :]
    chosen_logit = logits[records[target_step]['token_id']]
    model.zero_grad()
    chosen_logit.backward()
    
    if pixel_values.grad is None:
        print('  Warning: no gradient on pixel_values')
        return None
    
    grad = pixel_values.grad.detach().cpu().float()
    pix = pixel_values.detach().cpu().float()
    saliency = (grad * pix).abs().squeeze()
    if saliency.dim() == 3:
        saliency = saliency.mean(dim=0)
    elif saliency.dim() == 1:
        side = int(np.sqrt(saliency.shape[0] / 3))
        if side * side * 3 == saliency.shape[0]:
            saliency = saliency.reshape(3, side, side).mean(dim=0)
    return saliency.numpy()

In [ ]:
# Compute saliency for each coordinate's first digit
if coord_ranges:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    gt_match = re.search(r'<box>\((\d+),(\d+)\),\((\d+),(\d+)\)</box>', 
                         item['conversations'][1]['value'])
    if not gt_match:
        gt_match = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]',
                             item['conversations'][1]['value'])
    gt_coords = [int(gt_match.group(i)) for i in range(1, 5)] if gt_match else None
    
    for col, name in enumerate(['x1', 'y1', 'x2', 'y2']):
        ax = axes[col]
        ax.imshow(image)
        
        if name in coord_ranges:
            target_step = coord_ranges[name][0]
            print(f'Computing saliency for {name} (step {target_step})...')
            
            try:
                saliency = compute_pixel_saliency(model, processor, image, prompt_text, target_step, records)
                if saliency is not None:
                    sal_resized = np.array(Image.fromarray(saliency).resize(
                        (image.width, image.height), Image.BILINEAR))
                    ax.imshow(sal_resized, alpha=0.6, cmap='hot')
            except Exception as e:
                print(f'  Saliency failed for {name}: {e}')
        
        if gt_coords:
            gx1, gy1, gx2, gy2 = [c/1000 for c in gt_coords]
            rect = patches.Rectangle((gx1*image.width, gy1*image.height),
                (gx2-gx1)*image.width, (gy2-gy1)*image.height,
                lw=2, ec='lime', fc='none', ls='--')
            ax.add_patch(rect)
        
        ax.set_title(f'{name} pixel saliency', fontsize=12)
        ax.axis('off')
    
    fig.suptitle(f'Pixel Saliency (Gradient × Input): {prompt_text}', fontsize=11)
    plt.tight_layout()
    plt.show()